# 🧠 TRIBE v2 Playground: Colab launcher
1. **Runtime → Change runtime type → GPU** (T4 works; A100 is faster).
2. Add a Colab secret (🔑 sidebar) named `HF_TOKEN`: a Hugging Face *read* token (access to `meta-llama/Llama-3.2-3B` is optional: the app can use an ungated mirror of the same weights).
3. If this repo is private, add a secret `GH_TOKEN` (GitHub fine-grained token with read access to the repo).
4. Run all cells and open the `trycloudflare.com` link printed by the last cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata
try:
    auth = userdata.get('GH_TOKEN') + '@'
except Exception:
    auth = ''
%cd /content
if not os.path.exists('/content/nmai-capstone'):
    !git clone -q -b claude/pensive-faraday-gy9cjo https://{auth}github.com/gtougs/nmai-capstone.git
%cd /content/nmai-capstone
!git pull -q
!pip install -q uv -r requirements.txt "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2"
# tribev2 pins torch<2.7, which downgrades Colab's torch; realign torchaudio or transformers fails to import it
!pip install -q "torchaudio==$(python -c 'import torch; print(torch.__version__.split("+")[0])')"
!python -c "import torch, torchaudio; print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
from huggingface_hub import login, auth_check
login(token=userdata.get('HF_TOKEN'))
try:
    auth_check('meta-llama/Llama-3.2-3B'); print('✅ Official Llama 3.2 access OK')
except Exception:
    print('⚠️ No official Llama 3.2 access yet. In the app sidebar set "Text feature model" to "Ungated mirror (unsloth)". '
          'Note that video and audio need it too, because their speech is transcribed.')

In [ ]:
# Launch Streamlit, wait until it is healthy, then open a free Cloudflare quick tunnel (no account needed)
import subprocess, re, time, urllib.request
APP_DIR, LOG = '/content/nmai-capstone', '/content/streamlit.log'
subprocess.run('pkill -f "streamlit run"; pkill -f cloudflared', shell=True)  # clean restart if re-run
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared

app = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.headless', 'true', '--server.port', '8501'],
                       cwd=APP_DIR, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if urllib.request.urlopen('http://localhost:8501/_stcore/health', timeout=2).read() == b'ok': break
    except Exception:
        pass
    if app.poll() is not None: break
    time.sleep(1)
if app.poll() is not None:
    raise RuntimeError('Streamlit crashed on startup:\n' + open(LOG).read()[-3000:])
print('✅ Streamlit is healthy')

tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    if (m := re.search(r'https://[\w-]+\.trycloudflare\.com', line)):
        print('🧠 Open the playground (may take ~10 s to propagate):', m.group(0)); break
print(f'If the page breaks later, run:  !tail -50 {LOG}')
